# Trace one bid end-to-end — a `DETAILBOQAVAILABLE = 'Y'` tender

Unity Catalog: `ingestion_framework_test.bid_data_exploration`

Run 2 (`01_rfq.ipynb`) found `DETAILBOQAVAILABLE = 'Y'` on only 412 of 30,338 RFQs. Rather than judge that flag from aggregate counts alone, this notebook picks one real `Y`-flagged tender and follows it through every table — `rfq` → `rfqvendor` → `quotationline` → `altquotationline` → `docinfo`/`doclinks`/`vw_rfqvendor_documents` — to see what "has a detailed BOQ" actually looks like end to end. Contrast with `08_trace_bid_without_boq.ipynb` (D-111808, plus a `null`-flagged example).

## Step 1 — find a real candidate
Not just any `Y`-flagged RFQ — one with substantial `quotationline` data, so there's actually something to trace. Ordered by line count so the richest example surfaces first.

In [0]:
%sql
SELECT r.RFQNUM, r.DESCRIPTION, r.ORGID, r.ENTERDATE,
       COUNT(ql.QUOTATIONLINEID) AS line_count,
       COUNT(DISTINCT ql.VENDOR) AS vendor_count,
       COUNT(DISTINCT ql.BOQITEMNUM) AS distinct_boqitems
FROM ingestion_framework_test.bid_data_exploration.rfq r
LEFT JOIN ingestion_framework_test.bid_data_exploration.quotationline ql ON r.RFQNUM = ql.RFQNUM
WHERE r.DETAILBOQAVAILABLE = 'Y'
GROUP BY r.RFQNUM, r.DESCRIPTION, r.ORGID, r.ENTERDATE
ORDER BY line_count DESC
LIMIT 20

RFQNUM,DESCRIPTION,ORGID,ENTERDATE,line_count,vendor_count,distinct_boqitems
N-19304,Blanket Agreement for Maintenance of Lights and Small Power Systems for Substations & Pumping Stations,TRANSORG,2023-08-09T13:56:42Z,33824,7,4832
N-19042.1,"Blanket Agreement for Overhauling of Pumps for AD, AA & ADF Regions",TRANSORG,2023-02-03T11:58:03Z,7044,6,0
N-19829,"3 Years Blanket Agreement for UPS Systems, DC Chargers and Batteries related works for Water Network in all regions",TRANSORG,2024-07-04T16:11:59Z,5151,3,0
N-19539,Transformation and Upgrade of Telecom Network Technology from the current SDH to MPLS,TRANSORG,2025-05-26T12:53:09Z,4172,7,0
N-19991,"Blanket Agreement for Inspection Provision Services & Non Destructive Testing(NDT) services, to perform inspection of water static equipment in PS/RP/Piping network - 4 Regions",TRANSORG,2025-01-10T12:10:22Z,3990,3,1324
N-20127,400kV New Sweihan Switching Station Including Remote End Modifications Works For Integrating PV4,TRANSORG,2025-03-11T14:37:15Z,3654,7,0
N13719,Field instrumentations and actuators for Al Ain & NE Water assets for 2022,TRANSORG,2022-03-29T13:00:10Z,2759,31,0
N-19535,"400kV OHL Works for Haffar, ICAD4 & ICAD5 (SASN Retirement)",TRANSORG,2024-02-09T08:42:00Z,2480,4,0
N14070,2023 AA & NE WMD Spares for Field Instruments & Actuators,TRANSORG,2023-03-20T14:26:07Z,2016,24,0
N14454,2024 AA&NE Water O&M Spares for Field Instrumentations and Actuators.,TRANSORG,2024-03-28T09:04:36Z,1995,21,0


## Step 2 — set the RFQNUM to trace
Copy an `RFQNUM` from the candidates above into the widget this next cell creates (it'll appear at the top of the notebook), then run every cell below.

In [0]:
dbutils.widgets.text("rfqnum", "", "RFQNUM to trace")

### `rfq` — header

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfq WHERE RFQNUM = 'N-19535'

RFQNUM,DESCRIPTION,STATUS,STATUSDATE,ENTERDATE,ENTERBY,REPLYDATE,CLOSEONDATE,PURCHASEAGENT,RFQTYPE,REQUIREDDATE,REQUESTEDBY,SHIPTO,SHIPTOATTN,BILLTO,BILLTOATTN,REPLYTO,REPLYTOATTN,FOB,FREIGHTTERMS,SHIPVIA,PAYMENTTERMS,CHANGEBY,CHANGEDATE,PRIORITY,HISTORYFLAG,RFQ1,RFQ2,RFQ3,RFQ4,RFQ5,RFQ6,RFQ7,RFQ8,RFQ9,RFQ10,PRINTDATE,BUYERCOMPANY,ORGID,SITEID,RFQID,LANGCODE,HASLD,TDCOMPLETED,INSURANCEREV,TOTALAWVALUE,PMETHOD,VLSUBMSN,LEGALREV,AWCOMPLETED,TDAGENDA,FINANCEREV,TDMEETING,TDDATE,EXTCLOSEDATE,COMCONT,VLDCSN,WARRANTY,CONTYPE,AWDCSN,CONTPHONE,AWAGENDA,AWDATE,USEREV,FIXEDASSETS,TOTALCOST,TDDECISION,AWDECISION,ESTIMATEDCOST,AWSUBMSN,AWMEETING,DELVDEST,PROJCONS,CURRENCYCODE,INSPCLASS,AFENUM,ESTIMATEVAL,REPNAME,APPTYPE,TYPE,ROWSTAMP,REMARK,WFREPREMARKS,BIDSOPENDATE,KPIREMARKS,MAXINDIVIDUALVOL,TENDERDOCAPPRDATE,TENDERDOCSENDDATE,KPI1DATE,KPI2DATE,KPI3DATE,KPI4DATE,KPI5DATE,KPI6DATE,KPI17ATE,KPI8DATE,KPI9DATE,KPI10DATE,KPI7DATE,KPI11DATE,FIRSTDRAFT,SECONDRAFT,IITPACK,KPI,BIDBOND,PERFBOND,BENEFICIARY,CAPEX_CATEGORY,DURATIONUNIT,FUND_SOURCE,TYPE_BUSINESS,WARRANTYDAY,WARRANTYUNIT,CC,CLARIFICATIONDATE,DEALTBY,FORREVIEW,NOTE,PLEASECOMMENT,PLEASERECYCLE,PLEASEREPLY,PRINTDRAFTCOPY,PRINTESIGNATURE,REFNO,SELECTREPORT,SITEVISIT,TENDERDATE,URGENT,VENDOR,CBIDSOPENDATE,EOICLOSINGDATE,EOIEMAILSUBJECT,EOIEMAILTEXT,EOIREMARKS,EOIUSED,TENDERSTATUS,USEEBIDDING,EOIREFNO,EXTPRICECLOSEDATE,PRICECLOSEDATE,BIDEXTREFNO,INVITEREFNO,PIREFNO,EVALUATIONCUTOFF,TOTALSCORE,TRAINREQ,INSTALLREQ,SAMPLEREQ,CONCREQ,CALCREQ,TESTREPREQ,SPACKINGREQ,TECHDRAWREQ,TRAINING,SPACKDETAILS,TRAININGDAY,TRAININGUNIT,DOWNLOADTYPE,BIDBONDVALUE,EOIPUBLISHDATE,PUBLISHQUOTE,BIDBONDVALIDITY,CEILINGVALUE,RFQAUTHCODE,RFQCOMAUTHCODE,ADVANCEPAYMENT,ADVANCEPAYMENTPERCENT,DETAILBOQAVAILABLE,RETENTIONAPPLIED,RETENTIONPERCENT,COMEVALCUTOFF,COMTOTALSCORE,PARTIALDELIVERY,JUSTIFICATION,CPCSTARTDATE,CPCENDDATE,ECSTARTDATE,ECENDDATE,TECHSTART,TECHEND,COMMSTART,COMMEND,CPCSTART,CPCEND,BLREPDATE,BLSDDATE,AWARDREPDATE,AWARDSDDATE,TOTALAWVALUEWITHTAX,SUBWORKTYPE,CONSREQUIRED,PERFBANK,RETENMONEY,RPAC,RFAC,ADVANCEPAY,INVITEENTERED,INVITETEMP,EOICLOSED,BIDCLOSED,ECSTART,ECEND,COMPDISQUALIFIED,TETC_ENDDATE,AWTC_ENDDATE,TETC_STARTDATE,AWTC_STARTDATE,BUDGETBALANCE,BUDGETAVAILABLE,BUDGETCAT,ESTBUDGET,GUIDNUM,ITRTYPE,PROJECTREV,PROJECTID,ITRNUM,BUDGETCODE,APMASTERREV,APMASTERID,EXT_PROJECTID,EXT_ITRNUM,EXT_BUDGETCODE,EXT_PROJECTREV,POSTBID_DISCOUNT_CLOSEDATE,DISCOUNT_REVISION,EXT_POSTBID_DISCOUNT_CLOSEDATE,RECALLREPUBLISH,ALLOWAMENDMENT,N_PROJECT_NUMBER,N_TASK_NUMBER,UNDERREVIEW,ADPCBOARD,ADPCENDORSE,ADPCEO,ADPCHEC,ADPCREVIEWERS,ADPCTPC,ASTTEAM,ISPROJECT,ESTIMATECOST,ALLOWSINGLEIMPORT,SINGLESOURCE,MEMBERSFORIMP,BP_MASTERMILESTONE,ENDDATE,STARTDATE,PLANNEDSTARTDATE,PROCESSSTARTDATE,AWD_REMARKS,COMEVAL_REMARKS,TECHEVAL_REMARKS,FIRSTCBIDSOPENDATE,SHAREDWITHEWEC,NBFP,NBFPDONE,PROJLIFECYCLEID,EXTAADC1,DURATIONDAYS,CATEGORY,TDCOMMENTS,TDENDORSEDATE,TDENTERBY,TDRESULT,COMOPENDATE,COMOPENUSER,STAGE,TECHOPENDATE,TECHOPENUSER,AUTOROUTE,BUDGENHCOMMENTS,BUDGENHENDORSEDATE,BUDGENHENTERBY,BUDGENHRESULT,BLACOMMENTS,BLAENDORSEDATE,BLATDENTERBY,BLARESULT,BUDGENHFMDATE,BUDGENHFMENTERBY,BUDGENHMDDATE,BUDGENHMDENTERBY,BUDGENHSMDATE,BUDGENHSMENTERBY,BLASMDATE,BLASMENTERBY,SAMPLETESTREQ,BUDGENHCOMMENTSFIN,BUDGENHFINOFF,BUDGENHRESULTFIN,BUDGENHDEPTDEC,BUDGENHFMDEC,BUDGENHMDDEC,BUDGENHREJBY,BUDGENHREJDATE,BUDGENHREJREAASON,BUDGENHSMDEC,PROJECTDUR,PROJECTDURUNIT,TD,FU_STATUS,FU_SOURCE,FU_STATUSBY,FU_STATUSDATE,TECHEVALREQ,N_ISTTH
N-19535,"400kV OHL Works for Haffar, ICAD4 & ICAD5 (SASN Retirement)",CLOSE,2025-01-01T20:45:29Z,2024-02-09T08:42:00Z,NE015050,2024-07-01T11:00:00Z,2024-05-15T13:00:00Z,NE015050,null,2024-11-01T00:00:00Z,NE015048,null,MXINTADM,null,null,null,null,null,null,null,null,NE015050,2025-01-01T20:45:29Z,3.0000000000,1.0000000000,null,null,null,LS & UR,33,null,null,0E-10,null,null,2024-04-18T00:00:00Z,null,TRANSORG,TRANS,107886.0000000000,EN,0E-10,0E-10,null,307030751.4000,OPEN,null,null,0E-10,null,null,null,null,2024-07-01T0

### `rfqvendor` — invited vendors

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfqvendor WHERE RFQNUM = 'N-19535' ORDER BY VENDOR

RFQNUM,VENDOR,CONTACT,PHONE,FAXPHONE,EMAIL,CUSTOMERNUM,FOB,FREIGHTTERMS,SHIPVIA,PAYMENTTERMS,CURRENCYCODE,EXCHANGERATE,EXCHANGEDATE,BUYAHEAD,INTERNAL,REPLIEDDATE,GLCREDITACCT,INCLUSIVE1,INCLUSIVE2,INCLUSIVE3,INCLUSIVE4,INCLUSIVE5,VENDORQUOTENUM,ORGID,SITEID,RFQVENDORID,LANGCODE,HASLD,ROWSTAMP,BUYERREMARK,LINENUM,RFQV1,PRODUCTGROUPID,CURRENCYCODE2,RFQV3,RFQV4,DISCOUNT,SQR,MNGREMARK,ENTERDATE,RFQV2,OTHERCHG,RFQV5,SSUPREMARK,ENTEREDBY,ISVENINC,BYRRMK,SNSRMK,MGTRMK,BIDSTATUS,BIDSTATUSDATE,PASSWORD,PRICEIMPACT,REGRETREASON,RESETPASSWORD,CI,TCI,TI,USERID,VALIDTO,SUPPORTREMARKS,SUPPOERTCONTACTED,CIINITIATED,VALIDITYPERIOD,SCORE,SURVEYCREATED,BASETOTALAWARDCOST,TOTALAWARDCOST,TOTALDISCOUNT,TOTAWDWODISCOUNT,TOTBIDCOSTWDIS,TOTBIDCOSTWODIS,TOTALAWARDCOSTWITHTAX,COMPEVALTYPE,APPLYDISCOUNTBEFOREBIDS,APPLYDISCOUNTAFTERBIDS,DISCOUNT_PERCENT,APPLYDISCOUNT,DISCOUNTAFTERBIDS,APPLYDISCOUNTAFTERPI,DCI,DCIREQUIRED,DISCOUNTSTATUS,DISCOUNT_APPLIED_AFTERBIDS,DISCOUNT_APPLIED_AFTERPI,DISCOUNT_APPLIED_BEFOREBIDS,POSTBID_DISCOUNT_COUNTER,POSTBID_DISCOUNT_SENT,DISCOUNT_REVISION,DISCOUNT_SUBMISSION_DATE,TOTALAWARDCOSTWDIS,TOTALAWARDCOSTWITHTAXWDIS,DISCOUNT_APPLY_DATE,QUOTERENEWED,QUOTEVALIDITY,RFQV_EXTRA1,RFQV_EXTRA2,ISAWARDED,AGREECOC,WAIVEOWNERVAL
N-19535,001303,ALI KHALILI,0097126272322,0097126272748,cobrauae@emirates.net.ae,null,null,null,null,null,AED,1.0000000,2024-02-09T00:00:00Z,0E-10,0E-10,null,XG-0000-00000-251040-00-000000000-00-000000000,1.0000000000,1.0000000000,1.0000000000,0E-10,0E-10,null,TRANSORG,TRANS,548509.0000000000,EN,0E-10,13008511107,null,null,null,null,null,1.0000000000,null,null,null,null,2024-02-09T09:02:26Z,null,null,null,null,NE015050,0E-10,null,null,1.0000000000,COLLECTED,2024-04-19T23:52:07Z,null,0E-10,null,0E-10,0E-10,0E-10,0E-10,0013036KWQ,null,null,null,0E-10,null,null,0E-10,null,null,null,null,null,null,null,null,0E-10,0E-10,0.00,null,0E-10,0E-10,0E-10,0E-10,null,0E-10,0E-10,0E-10,null,0E-10,null,null,null,null,null,0E-10,null,null,null,0E-10,0E-10,0E-10
N-19535,001669,Shakun Goyal,0097165172900 Ext-210,0097126729873,ltcgad@emirates.net.ae,null,null,null,null,null,AED,1.0000000,2024-02-09T00:00:00Z,0E-10,0E-10,null,XG-0000-00000-251040-00-000000000-00-000000000,1.0000000000,1.0000000000,1.0000000000,0E-10,0E-10,null,TRANSORG,TRANS,548510.0000000000,EN,0E-10,13016203727,null,null,null,null,null,1.0000000000,null,null,null,null,2024-02-09T09:02:26Z,null,null,null,null,NE015050,0E-10,null,null,2.0000000000,COLLECTED,2024-04-22T08:49:31Z,null,0E-10,null,0E-10,0E-10,0E-10,0E-10,0016697WC1,null,null,null,0E-10,null,null,0E-10,null,null,null,null,null,null,null,null,0E-10,0E-10,0.00,null,0E-10,0E-10,0E-10,0E-10,null,0E-10,0E-10,0E-10,null,0E-10,null,null,null,null,null,0E-10,null,null,null,0E-10,0E-10,0E-10
N-19535,002153,GABRIEL VASILACHE,0097126732331,0097126732336,office.emirates@electromontaj.ro,null,null,null,null,null,AED,1.0000000,2024-02-09T00:00:00Z,0E-10,0E-10,null,XG-0000-00000-251040-00-000000000-00-000000000,1.0000000000,1.0000000000,1.0000000000,0E-10,0E-10,null,TRANSORG,TRANS,548511.0000000000,EN,0E-10,13001356793,null,null,null,null,null,1.0000000000,null,null,null,null,2024-02-09T09:02:26Z,null,null,null,null,NE015050,0E-10,null,null,3.0000000000,NOT SIGNED,2024-04-18T14:24:30Z,null,0E-10,null,0E-10,0E-10,0E-10,0E-10,00215301NO,null,null,null,0E-10,null,null,0E-10,null,null,null,null,null,null,null,null,0E-10,0E-10,null,null,0E-10,0E-10,0E-10,0E-10,null,0E-10,0E-10,0E-10,null,0E-10,null,null,null,null,null,0E-10,null,null,null,0E-10,0E-10,0E-10
N-19535,003235,Subhash Zinjad,0097125550801,0097125550816,inabensa@abengoa.com,null,null,null,null,null,AED,1.0000000,2024-02-09T00:00:00Z,0E-10,0E-10,2024-07-01T10:55:36Z,XG-0000-00000-251040-00-000000000-00-000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000,1.0000000000,INA-GEN-24-1813T&C,TRANSORG,TRANS,548512.0000000000,EN,0E-10,14216036755,null,null,null,null,null,1.0000000000,null,null,null,null,2024-02-09T09:02:26Z,null,null,null,null,NE015050,0E-10,null

### `quotationline` — the itemized BOQ itself (the whole point of this notebook)

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.quotationline WHERE RFQNUM = 'N-19535' ORDER BY VENDOR, BOQITEMNUM, RFQLINENUM

RFQNUM,RFQLINENUM,VENDOR,QUOTATIONLINEID,ITEMNUM,MANUFACTURER,MODELNUM,ORDERQTY,ORDERUNIT,UNITCOST,LINECOST,EOQ,DELIVERYTIME,DELIVERYDATE,ENTERDATE,ENTERBY,ISAWARDED,SELECTEDFORDISPLAY,GLCREDITACCT,TAX1CODE,TAX1,TAX2CODE,TAX2,TAX3CODE,TAX3,TAX4CODE,TAX4,TAX5CODE,TAX5,CATALOGCODE,MEMO,DESCRIPTION,QUOTESTARTDATE,QUOTEENDDATE,LINECOST2,VENDORPACKCODE,VENDORPACKQUANTITY,VENDORWAREHOUSE,SITEID,ORGID,LINETYPE,ITEMSETID,CONDITIONCODE,COMMODITYGROUP,COMMODITY,LANGCODE,HASLD,MKTPLCITEM,ROWSTAMP,AWARDCOST,ISALTAWARDED,ALTAWARDEDLINE,QL1,QL5,QL4,QL3,SUGAWARD,QL2,CONVERSION,LOADEDCOST,ISEBID,ISPIAWARDED,PRICEIMPACT,BOQITEMNUM,APPLYDISCOUNTAFTERBIDS,DISCOUNT_PERCENT,DISCOUNT_UNITCOST,DISCOUNT_APPLIED_AFTERBIDS,LINECOSTWDIS,QL_EXTRA1,WARRANTYDURATION,WARRANTYPERITEM,WARRANTYREQ
N-19535,1.0000000000,003235,2150603329.0000000000,null,null,null,1.00,HEADER,null,null,0.00,null,null,2024-04-23T17:03:15Z,0032357RPV,0E-10,1.0000000000,XN-0000-00000-251040-00-000000000-00-000000000,IFTR,0.0000,null,0.00,null,0.00,null,0.00,null,0.00,null,null,Section : 1 - LILO OF 400kV D/C OHL PVDF (PV2) - PVAJ (PV3) to HFRG,null,null,null,null,null,null,TRANS,TRANSORG,SERVICE,null,null,null,null,EN,0E-10,0E-10,13464370078,null,0E-10,null,null,null,null,null,0E-10,NOQUOTE,null,0.0000,0E-10,0E-10,0E-10,null,0E-10,0.00,null,0E-10,0.0000,null,null,null,null
N-19535,2.0000000000,003235,2150603331.0000000000,null,null,null,13.00,KM,null,null,0.00,null,null,2024-04-23T17:03:15Z,0032357RPV,0E-10,1.0000000000,XN-0000-00000-251040-00-000000000-00-000000000,IFTR,0.0000,null,0.00,null,0.00,null,0.00,null,0.00,null,null,Standard/Conventional Survey (ERECTION PRICE / LOCAL),null,null,null,null,null,null,TRANS,TRANSORG,SERVICE,null,null,null,null,EN,0E-10,0E-10,13464370100,null,0E-10,null,null,null,null,null,0E-10,NOQUOTE,null,0.0000,0E-10,0E-10,0E-10,null,0E-10,0.00,null,0E-10,0.0000,null,null,null,null
N-19535,3.0000000000,003235,2150603333.0000000000,null,null,null,6.00,KM,null,null,0.00,null,null,2024-04-23T17:03:15Z,0032357RPV,0E-10,1.0000000000,XN-0000-00000-251040-00-000000000-00-000000000,IFTR,0.0000,null,0.00,null,0.00,null,0.00,null,0.00,null,null,4 Meter width x 0.5 Meter Thick Gatch Road (ERECTION PRICE / LOCAL),null,null,null,null,null,null,TRANS,TRANSORG,SERVICE,null,null,null,null,EN,0E-10,0E-10,13464370102,null,0E-10,null,null,null,null,null,0E-10,NOQUOTE,null,0.0000,0E-10,0E-10,0E-10,null,0E-10,0.00,null,0E-10,0.0000,null,null,null,null
N-19535,4.0000000000,003235,2150603335.0000000000,null,null,null,1.00,KM,null,null,0.00,null,null,2024-04-23T17:03:15Z,0032357RPV,0E-10,1.0000000000,XN-0000-00000-251040-00-000000000-00-000000000,IFTR,0.0000,null,0.00,null,0.00,null,0.00,null,0.00,null,null,4 Meter width x 1.0 Meter Thick Gatch Road (ERECTION PRICE / LOCAL),null,null,null,null,null,null,TRANS,TRANSORG,SERVICE,null,null,null,null,EN,0E-10,0E-10,13464370104,null,0E-10,null,null,null,null,null,0E-10,NOQUOTE,null,0.0000,0E-10,0E-10,0E-10,null,0E-10,0.00,null,0E-10,0.0000,null,null,null,null
N-19535,5.0000000000,003235,2150603337.0000000000,null,null,null,30.00,NO,null,null,0.00,null,null,2024-04-23T17:03:15Z,0032357RPV,0E-10,1.0000000000,XN-0000-00000-251040-00-000000000-00-000000000,IFTR,0.0000,null,0.00,null,0.00,null,0.00,null,0.00,null,null,20 Meter depth (ERECTION PRICE / LOCAL),null,null,null,null,null,null,TRANS,TRANSORG,SERVICE,null,null,null,null,EN,0E-10,0E-10,13464370106,null,0E-10,null,null,null,null,null,0E-10,NOQUOTE,null,0.0000,0E-10,0E-10,0E-10,null,0E-10,0.00,null,0E-10,0.0000,null,null,null,null
N-19535,6.0000000000,003235,2150603339.0000000000,null,null,null,9.00,NO,null,null,0.00,null,null,2024-04-23T17:03:15Z,0032357RPV,0E-10,1.0000000000,XN-0000-00000-251040-00-000000000-00-000000000,IFTR,0.0000,null,0.00,null,0.00,null,0.00,null,0.00,null,null,30 Meter depth (ERECTION PRICE / LOCAL),null,null,null,null,null,null,TRANS,TRANSORG,SERVICE,null,null,null,null,EN,0E-10,0E-10,13464370108,null,0E-10,null,null,null,null,null,0E-10,NOQUOTE,null,0.0000,0

### `altquotationline` — any alternates offered

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.altquotationline WHERE RFQNUM = 'N-19535' ORDER BY VENDOR, RFQLINENUM

TAX1,TAX3,TAX2,ALTQUOTATIONLINEID,QUOTEENDDATE,TAX5,SELECTEDFORDISPLAY,MANUFACTURERNAME,AWARDCOST,ALTQUOTLINEUID,QUOTESTARTDATE,ENTERDATE,UNITCOST,MEMO,ISAWARDED,LINENUM,TAX4,LINECOST2,ENTERBY,DELIVERYDATE,LINECOST,SUGAWARD,SERVICE,GLCREDITACCT,VENDOR,MANUFACTURER,CATALOGCODE,CONVERSION,DELIVERYTIME,MODELNUM,ORDERQTY,EOQ,DESCRIPTION,ITEMNUM,ORDERUNIT,MKTPLCITEM,VENDORPACKCODE,VENDORPACKQUANTITY,VENDORWAREHOUSE,ORGID,QL1,QL2,QL3,QL4,QL5,RFQNUM,RFQLINENUM,SITEID,TAX3CODE,TAX5CODE,TAX4CODE,TAX1CODE,TAX2CODE,ROWSTAMP,HASLD,ITEMSETID,LINETYPE,LOADEDCOST,ISEBID,PRICEIMPACT,BOQITEMNUM,DISCOUNT_PERCENT,APPLYDISCOUNTAFTERBIDS,DISCOUNT_APPLIED_AFTERBIDS,LINECOSTWDIS


### `docinfo` / `doclinks` / `vw_rfqvendor_documents` — attached documents

Notebook 05 is still deliberately deferred as a general investigation, but it's worth checking directly here for one specific tender: does a real bid document exist and link back to this RFQNUM? Schemas are unknown so far (notebook 05 was never run) — start with `DESCRIBE`.

In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.docinfo

col_name,data_type,comment
DOCUMENT,varchar(50),null
DESCRIPTION,varchar(254),null
APPLICATION,varchar(8),null
STATUS,varchar(8),null
STATUSDATE,timestamp,null
CREATEDATE,timestamp,null
REVISION,"decimal(38,10)",null
CHANGEBY,varchar(31),null
CHANGEDATE,timestamp,null
DOCLOCATION,varchar(10),null


In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.doclinks

col_name,data_type,comment
DOCUMENT,varchar(50),null
OWNERTABLE,varchar(30),null
OWNERID,"decimal(38,10)",null
REFERENCE,varchar(8),null
DOCTYPE,varchar(16),null
DOCVERSION,varchar(20),null
GETLATESTVERSION,"decimal(38,10)",null
CREATEBY,varchar(31),null
CREATEDATE,timestamp,null
CHANGEBY,varchar(31),null


In [0]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents

col_name,data_type,comment
rfqnum,varchar(20),null
vendor,varchar(44),null
bidstatus,varchar(25),null
rfqvendorid,"decimal(38,10)",null
ownertable,varchar(30),null
docinfoid,"decimal(38,10)",null
document,varchar(50),null
urlname,varchar(250),null


Once the schemas above are visible, uncomment and fix the column name below (guessing `RFQNUM`, following the convention every other table uses — may need correcting):

In [0]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents WHERE RFQNUM = 'N-19535'

rfqnum,vendor,bidstatus,rfqvendorid,ownertable,docinfoid,document,urlname
N-19535,99471778,SUBMITTED,548516.0000000000,RFQVENDOR,9224796.0000000000,6. MANPOWER HISTOGRAM & EQUIPMENT DEPLOYMENT,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\TECHBID\ManpowerHistogram1719809238637.pdf
N-19535,99471778,SUBMITTED,548516.0000000000,RFQVENDOR,9224887.0000000000,6.6 NARRATIVE ON PROGRAM ASSUMPTIONS,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\TECHBID\WorkExecutionstrategy1719810399391.pdf
N-19535,003235,SUBMITTED,548512.0000000000,RFQVENDOR,9222144.0000000000,FOLDER NO. 4.3 VENDOR DOCUMENTS,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\TECHBID\FolderNo.4.3VendorDocuments.pdf
N-19535,003235,SUBMITTED,548512.0000000000,RFQVENDOR,9222148.0000000000,FOLDER NO. 4.5 VENDOR DOCUMENTS,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\TECHBID\FolderNo.4.5VendorDocuments.pdf
N-19535,003235,SUBMITTED,548512.0000000000,RFQVENDOR,9222158.0000000000,FOLDER NO. 5.4 TENDER DOC SIGNED,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\TECHBID\FolderNo.5.4Tenderdocsigned.pdf
N-19535,003235,SUBMITTED,548512.0000000000,RFQVENDOR,9225893.0000000000,TENDER BOND N-19535-AED 400K,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\BIDBOND\TenderbondN-19535-AED400k.pdf
N-19535,003300,SUBMITTED,548513.0000000000,RFQVENDOR,9219565.0000000000,17.4 - HW FITTINGS - DALEKOVOD,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\TECHBID\17.4-HWFITTINGS-DALEKOVOD.pdf
N-19535,003300,SUBMITTED,548513.0000000000,RFQVENDOR,9219574.0000000000,18 - TENDER DOCUMENT 02,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\TECHBID\18-TenderDocument02.pdf
N-19535,99471778,SUBMITTED,548516.0000000000,RFQVENDOR,9224521.0000000000,3. FORM OF TENDER SECURITY,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\TECHBID\TENDERNO19535-299T401241630001.pdf
N-19535,99471778,SUBMITTED,548516.0000000000,RFQVENDOR,9239838.0000000000,BID BOND,\\advapfsn02\LinkedDocs\DoclinkProd\doclinks\EBID\BIDBOND\Bidbond1719802780525.pdf


**Observations:**
- _(fill in: which RFQNUM did you trace, and what does a genuinely populated BOQ look like — is `BOQITEMNUM` a real per-item code here, or still a coarse section label like the Run 2 examples? Does `LINETYPE` split CIF/Erection for this one? Are documents actually attached and retrievable?)_